In [ ]:
# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

zsh:1: command not found: wget


In [2]:
with open('input.txt', 'r', encoding='utf-8') as f: 
    text = f.read()

len(text)

1115393

In [3]:
# create a set of all characters that appear, then turn it into a sorted list
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [4]:
# tokenize the text
# very simple mapping using the index of the character (sorted based on the ASCII value of each character)

stoi = {char : i for i, char in enumerate(chars)}
itos = {i : char for i, char in enumerate(chars)}

# print(stoi)
# print(itos)

# encode = lambda char : stoi[char]
# decode = lambda i : itos[i]
encode = lambda s : [stoi[char] for char in s] # take a string and turn it into a list of integers
decode = lambda l : ''.join([itos[i] for i in l]) # take a list of integers and turn it into a string

print(encode("wowowwowowhello!"))
print(decode(encode('wowowwowowhello')))

[61, 53, 61, 53, 61, 61, 53, 61, 53, 61, 46, 43, 50, 50, 53, 2]
wowowwowowhello


In [5]:
import torch 
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115393]) torch.int64


In [6]:
# train/val split
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

train_data[:10]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])

In [7]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]

for i in range(block_size): 
    context = x[:i+1]
    target = y[i]
    print(f"when the input is {context} the target is: {target}")

when the input is tensor([18]) the target is: 47
when the input is tensor([18, 47]) the target is: 56
when the input is tensor([18, 47, 56]) the target is: 57
when the input is tensor([18, 47, 56, 57]) the target is: 58
when the input is tensor([18, 47, 56, 57, 58]) the target is: 1
when the input is tensor([18, 47, 56, 57, 58,  1]) the target is: 15
when the input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
when the input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [8]:
torch.manual_seed(1337)

batch_size = 4 # number of independent sequences we will be processing in parallel
block_size = 8 # size of the chunks or maximum context length

def get_batches(split): 
    data = train_data if split == "train" else val_data
    ix = torch.randint((len(data) - block_size), (batch_size,)) # batch_size number of random integer from 0 to len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix]) # take the 1d tensors previously and stack them
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batches("train")
print(f"size of inputs: {xb.shape} \ninputs: {xb}")
print(f"size of ouputs: {yb.shape} \noutputs: {yb}")

print("---------")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b][:t+1]
        target = yb[b][t]
        print(f"the context is: {context.tolist()} and the target is: {target}")

size of inputs: torch.Size([4, 8]) 
inputs: tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
size of ouputs: torch.Size([4, 8]) 
outputs: tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
---------
the context is: [53] and the target is: 59
the context is: [53, 59] and the target is: 6
the context is: [53, 59, 6] and the target is: 1
the context is: [53, 59, 6, 1] and the target is: 58
the context is: [53, 59, 6, 1, 58] and the target is: 56
the context is: [53, 59, 6, 1, 58, 56] and the target is: 47
the context is: [53, 59, 6, 1, 58, 56, 47] and the target is: 40
the context is: [53, 59, 6, 1, 58, 56, 47, 40] and the target is: 59
the context is: [49] and the target is: 43
the context is: [49, 43] and the target is: 43
the context is: [49, 43, 43] a

In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module): 
    def __init__(self, vocab_size): 
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None): 
        # both idx and targets are of dimension (B, T) tensors
        logits = self.token_embedding_table(idx) #(B, T, C)

        if targets is None: 
            loss = None
        else:  
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens): 
        # idx is (B, T) array of indices in the current context 
        for _ in range(max_new_tokens): 
            logits, loss = self(idx) # get the predictions
            logits = logits[:, -1, :] # grabs the last element in the T (time) dimension, becomes (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(f"Shape of the logits tensor is: {logits.shape}")
print(f"Shape of loss tensor is: {loss.shape}, and the value is: {loss}")

idx = torch.zeros((1, 1), dtype=torch.long) # creating a (1, 1) tensor filled with 0 of type int
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist())) # need [0] to get individual batches of tensors

Shape of the logits tensor is: torch.Size([32, 65])
Shape of loss tensor is: torch.Size([]), and the value is: 4.894842624664307

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [10]:
# creating a PyTorch optimizer 
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [11]:
batch_size = 32

for step in range(10000): 
    xb, yb = get_batches("train")

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) 
    loss.backward()
    optimizer.step()

print(loss.item())

2.4782700538635254


In [12]:
print(decode(m.generate(idx, max_new_tokens=400)[0].tolist()))


llo br. ave aviasurf my, may be ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofoff ve!
RIfans picsheserer hee anf,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
By bre ndy; by s afreanoo adicererupa anse tecorro llaus a!
OLengerithesinthengove fal ames trr
TI ar I t, mes, n sar; my w,

Whank'the
THek' merer,


In [13]:
m.token_embedding_table.weight

Parameter containing:
tensor([[ 2.4508, -5.1098, -5.3856,  ..., -3.5092, -1.8304, -5.8375],
        [-4.1185, -3.9636, -5.1189,  ..., -6.4554,  0.5548, -3.1749],
        [ 2.7876,  2.1527, -2.8951,  ..., -3.9781, -4.4770, -2.4746],
        ...,
        [-0.7903, -1.0172, -3.6450,  ..., -2.2084, -0.6676, -2.0405],
        [-0.4019,  2.8263, -1.3090,  ..., -3.7432, -4.9483, -4.8316],
        [-3.5922, -0.9298, -4.3614,  ..., -2.7886,  0.2870,  0.0539]],
       requires_grad=True)

# The mathematical trick in self attention
Toy example

In [14]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2 # batch, time, channels
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

In [15]:
# math trick
torch.manual_seed(42)
a = torch.tril(torch.ones((3, 3)))
a = a / torch.sum(a, 1, keepdim=True) # keepdim=True makes sure that the matrix that is created from torch.sum doesn't get collapsed
b = torch.randint(0, 10, (3, 2)).float()
c = a @ b

print(f"a={a}")
print(f"b={b}")
print(f"c={c}")

a=tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c=tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


Averaging past context. 
This is weak and lossy, but the concept applies

In [16]:
# version 1: for loop to average the past context

# we want x[b, t] = mean_{i <= t} x_{b, i}
xbow = torch.zeros((B, T, C)) # x bag of words (xbow)
for b in range(B): 
    for t in range(T): 
        xprev = x[b, :t+1] # of dimension (t, C) 
        xbow[b, t] = torch.mean(xprev, 0)

In [17]:
# version 2: using matrix multiplication

wei = torch.tril(torch.ones(T, T)) # size (T, T)
wei = wei / torch.sum(wei, 1, keepdim=True)
# Pytorch adds a batch dimension to wei and it becomes (B, T, T)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) --> (B, T, C)
torch.allclose(xbow, xbow2)

True

In [18]:
# version 3: softmax 
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

True

In [21]:
# version 4: self attention!
torch.manual_seed(1337)
B, T, C = 4, 8, 32 # batch, time, channel
x = torch.randn(B, T, C)

# single head of self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False) 
query = nn.Linear(C, head_size, bias=False) 
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B, T, head_size)
q = query(x) # (B, T, head_size)
wei = k @ q.transpose(-2, -1) * (head_size ** -0.5) # (B, T, head_size) @ (B, head_size, T) --> (B, T, T)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T, T)) wei is no longer zeros, they are coming from the dot product of keys and the queries
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
# out  = wei @ x

out.shape

torch.Size([4, 8, 16])

In [22]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5221, 0.4779, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3602, 0.3210, 0.3188, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2980, 0.4039, 0.1578, 0.1404, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1643, 0.1243, 0.1678, 0.1865, 0.3570, 0.0000, 0.0000, 0.0000],
        [0.2656, 0.2110, 0.1137, 0.1214, 0.2018, 0.0865, 0.0000, 0.0000],
        [0.1761, 0.1327, 0.1371, 0.0974, 0.1476, 0.1918, 0.1173, 0.0000],
        [0.1046, 0.1260, 0.0922, 0.0906, 0.1476, 0.1588, 0.1432, 0.1371]],
       grad_fn=<SelectBackward0>)